In [ ]:
!pip install torchmetrics -q

In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import torchmetrics
import numpy as np
import cv2
from google.colab.patches import cv2_imshow

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git
!cp dinov3/hubconf.py .

In [ ]:
import sys
sys.path.append("dinov3")

In [ ]:
!wget http://www.agentspace.org/download/mydinov3.zip
!unzip -P dino mydinov3.zip

In [ ]:
backbone = torch.hub.load('.', 'dinov3_vits16', source='local', weights='dinov3_vits16_pretrain_lvd1689m.pth') # dinov3_vits16
backbone.to(device)
backbone.eval()

In [ ]:
def make_transform(resize_size = 768):
    to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([to_tensor, resize, normalize])

img_size = 768
transform = make_transform(img_size)

In [ ]:
!pip install fastncut

In [ ]:
from fastncut import ncut, toCosSin, extendWithFix

In [ ]:
!wget http://agentspace.org/download/cup845972797.png

In [ ]:
image = cv2.imread("cup845972797.png")

In [ ]:
cv2_imshow(image)

In [ ]:
def cv2_imshow_mask(image, mask):
    overlay = image.copy()
    overlay[mask > 0] = (0,255,255)
    result = cv2.addWeighted(overlay, 0.4, image, 0.6, 0)
    cv2_imshow(result)

In [ ]:
def fix_polarity(binary):
    corners = np.array([ binary[0,0], binary[0,-1], binary[-1,0], binary[-1,-1] ], np.float32) / 255
    if np.average(corners) < 0.5:
        return binary
    else:
        print("inverted")
        return ~binary

Our method

In [ ]:
with torch.no_grad():
    blob = transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
    feats = backbone.get_intermediate_layers(blob,norm=False)[0]
    feats = feats.view(feats.shape[0],blob.shape[2]//backbone.patch_size,blob.shape[3]//backbone.patch_size,feats.shape[-1])
    feats = feats.permute(0,3,1,2)
    feats = F.interpolate(feats, size=(image.shape[0], image.shape[1]), mode="bilinear", align_corners=False)
    bipartition0 = ncut(feats, data_format="bchw", num_iters=4)[0]
    bipartition = bipartition0.cpu().numpy().astype(np.uint8)*255

In [ ]:
feats.min(), feats.max()

In [ ]:
feats.shape

In [ ]:
cv2_imshow_mask(image, fix_polarity(bipartition))

In [ ]:
bipartition1 = ncut(feats, data_format="bchw", num_iters=4, mask=~bipartition0)[0]
bipartition = bipartition1.cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image, fix_polarity(bipartition))

In [ ]:
bipartition2 = ncut(feats, data_format="bchw", num_iters=4, mask=bipartition1)[0]
bipartition = bipartition2.cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image, fix_polarity(bipartition))

In [ ]:
bipartition = (bipartition1 & ~bipartition2).cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image, fix_polarity(bipartition))

Our method without interpolation (lower resolution requires more iterations)

In [ ]:
with torch.no_grad():
    blob = transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
    feats = backbone.get_intermediate_layers(blob,norm=False)[0]
    feats = feats.view(feats.shape[0],blob.shape[2]//backbone.patch_size,blob.shape[3]//backbone.patch_size,feats.shape[-1])
    bipartition = ncut(feats, data_format="bhwc", num_iters=8)[0].cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image,cv2.resize(fix_polarity(bipartition),(image.shape[1],image.shape[0]),interpolation=cv2.INTER_NEAREST))

Comparision with the original algorithm (Shi & Malik 2000, Wang et. al 2023)

In [ ]:
with torch.no_grad():
    blob = transform(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
    feats = backbone.get_intermediate_layers(blob,norm=False)[0][0]

In [ ]:
feats.shape

In [ ]:
48**2

In [ ]:
feats.norm(dim=1)

In [ ]:
feats

In [ ]:
W = feats @ feats.t()

In [ ]:
W.shape, W.min(), W.max()

- without thresholding

In [ ]:
D = torch.diag(W.sum(dim=1))

In [ ]:
torch.lobpcg(A=D-W, k=1, B=D, largest=True)[0]

In [ ]:
eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)

In [ ]:
eigenvalues

In [ ]:
eigenvectors[:,1]

In [ ]:
bipartition = (eigenvectors[:,1] > 0).view(48,48).cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image,cv2.resize(fix_polarity(bipartition),(image.shape[1],image.shape[0]),interpolation=cv2.INTER_NEAREST))

- with thresholding

In [ ]:
WW = W.clone()
WW[WW<=0.0] = 1e-5
DD = torch.diag(WW.sum(dim=1))

In [ ]:
eigenvalues, eigenvectors = torch.lobpcg(A=DD-WW, k=2, B=DD, largest=False)

In [ ]:
eigenvalues

In [ ]:
bipartition = (eigenvectors[:,1] > 0).view(48,48).cpu().numpy().astype(np.uint8)*255

In [ ]:
cv2_imshow_mask(image,cv2.resize(fix_polarity(bipartition),(image.shape[1],image.shape[0]),interpolation=cv2.INTER_NEAREST))